# V28 Colab — fold-0 train only

V27 **0.914** stays selected. No `submission.csv`.

Runtime → **T4 GPU**. Cache comes from Kaggle → `/content/v28_cache`. Checkpoints → Drive `rsna-knee-v28/checkpoints/fold0`. Clone folds 1–4 only if fold-0 val **≥ 0.83**.


In [ ]:
import sys
import torch

print("python", sys.version.split()[0], "torch", torch.__version__)
print("cuda", torch.cuda.is_available())
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("gpu", gpu)
if torch.cuda.is_available():
    name = gpu.upper()
    assert not any(b in name for b in ("H100", "H200", "B200", "A100")), gpu
    print("ok", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GiB")
else:
    raise SystemExit("Runtime -> Change runtime type -> T4 GPU")


python 3.12.13 torch 2.11.0+cu128
cuda True
gpu Tesla T4
ok 14.56 GiB


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
CKPT = Path("/content/drive/MyDrive/rsna-knee-v28/checkpoints/fold0")
CKPT.mkdir(parents=True, exist_ok=True)
print("ckpt", CKPT)


Mounted at /content/drive
ckpt /content/drive/MyDrive/rsna-knee-v28/checkpoints/fold0


In [ ]:
%pip -q install "transformers==4.46.3" safetensors scikit-learn kaggle
import transformers, kaggle
print("transformers", transformers.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.46.3


In [ ]:
from pathlib import Path
print("kaggle token on runtime", Path("/root/.kaggle/access_token").is_file())


kaggle token written True


In [ ]:
from pathlib import Path
import json
import os

tok = Path("/root/.kaggle/access_token")
assert tok.is_file(), "kaggle token missing on runtime"
os.environ["KAGGLE_API_TOKEN"] = tok.read_text(encoding="utf-8").strip()
!kaggle kernels status danielsolo1770/rsna-knee-v28-cache-224
!kaggle kernels output danielsolo1770/rsna-knee-v28-cache-224 -p /content/v28_kaggle_out -o
hits = list(Path("/content/v28_kaggle_out").rglob("manifest.json"))
print("manifests", hits)
assert hits, "no manifest in kernel output"
CACHE = hits[0].parent
man = json.loads(hits[0].read_text())
print(man)
assert man.get("status") == "complete" and int(man["n"]) == 4407, man
for name in ("x_f16.dat", "mask.dat", "y.npy", "uids.txt", "manifest.json"):
    p = CACHE / name
    print(name, p.is_file(), p.stat().st_size if p.is_file() else 0)
    assert p.is_file(), name
print("CACHE", CACHE)


danielsolo1770/rsna-knee-v28-cache-224 has status "KernelWorkerStatus.COMPLETE"
Output file downloaded to /content/v28_kaggle_out/v28_cache/manifest.json
Output file downloaded to /content/v28_kaggle_out/v28_cache/mask.dat
Output file downloaded to /content/v28_kaggle_out/v28_cache/progress.json
Output file downloaded to /content/v28_kaggle_out/v28_cache/uids.txt
Output file downloaded to /content/v28_kaggle_out/v28_cache/x_f16.dat
Output file downloaded to /content/v28_kaggle_out/v28_cache/y.npy
Kernel log downloaded to /content/v28_kaggle_out/rsna-knee-v28-cache-224.log 
manifests [PosixPath('/content/v28_kaggle_out/v28_cache/manifest.json')]
{'n': 4407, 'x_shape': [4407, 3, 3, 224, 224], 'dtype': 'float16', 'img': 224, 'crop_mm': 160.0, 'recipe': 'v28-224-3slice-3fs-planes', 'targets': ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'], 'n_ok_last_pass': 4407, 'seconds': 5586.06253671

In [ ]:
from pathlib import Path

Path("/content/train_core.py").write_text(open("/content/train_core.py").read() if False else r'''"""Resumable V28 DINOv2-small partial-FT. Cache-only (no DICOM on GPU)."""
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_SLOT = 3
N_SLICE = 3
IMG = 224
SEED = 20260817
UNFREEZE_LAST = 4
AUX_W = 0.25
N_FOLDS = 5


def log(msg: str, t0: float) -> None:
    print(f"[{time.time() - t0:7.1f}s] {msg}", flush=True)


def report_group(uid: str) -> str:
    return hashlib.md5(uid.encode()).hexdigest()[:8]


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class TrainCfg:
    cache_dir: str
    out_dir: str
    fold: int = 0
    epochs: int = 12
    batch: int = 16
    lr_head: float = 2e-4
    lr_backbone: float = 2e-5
    weight_decay: float = 1e-4
    unfreeze_last: int = UNFREEZE_LAST
    patience: int = 4
    resume: bool = True
    smoke: bool = False
    num_workers: int = 2
    seed: int = SEED
    dino_id: str = "facebook/dinov2-small"
    augment: bool = True
    mixup: float = 0.2
    dropout: float = 0.35


class CacheDS(Dataset):
    def __init__(self, x, mask, y, uids, idx, augment=False):
        self.x = x
        self.mask = mask
        self.y = y
        self.uids = uids
        self.idx = np.asarray(idx, dtype=np.int64)
        self.augment = bool(augment)

    def __len__(self):
        return int(len(self.idx))

    def __getitem__(self, i):
        j = int(self.idx[i])
        x = np.asarray(self.x[j], dtype=np.float32)
        m = np.asarray(self.mask[j], dtype=np.float32)
        y = np.asarray(self.y[j], dtype=np.float32)
        if self.augment:
            if random.random() < 0.5:
                x = np.ascontiguousarray(x[..., :, ::-1])
            if random.random() < 0.7:
                x = np.clip(x * np.float32(random.uniform(0.8, 1.2)), 0.0, 1.0)
            if random.random() < 0.4:
                x = np.clip(x + np.float32(random.uniform(-0.08, 0.08)), 0.0, 1.0)
        return torch.from_numpy(x), torch.from_numpy(m), torch.from_numpy(y), self.uids[j]


def load_cache(cache_dir: Path):
    man_p = cache_dir / "manifest.json"
    if not man_p.is_file():
        raise FileNotFoundError(f"cache missing at {cache_dir}")
    man = json.loads(man_p.read_text(encoding="utf-8"))
    n = int(man["n"])
    shape = tuple(man["x_shape"])
    x = np.memmap(cache_dir / "x_f16.dat", dtype=np.float16, mode="r", shape=shape)
    mask = np.memmap(cache_dir / "mask.dat", dtype=np.uint8, mode="r", shape=(n, N_SLOT))
    y = np.load(cache_dir / "y.npy")
    uids = (cache_dir / "uids.txt").read_text(encoding="utf-8").splitlines()
    if len(uids) != n or y.shape[0] != n:
        raise ValueError(f"cache length mismatch n={n} uids={len(uids)} y={y.shape}")
    return man, x, mask, y, uids


class PartialFT(nn.Module):
    def __init__(self, dino_id: str, unfreeze_last: int, dropout: float = 0.35):
        super().__init__()
        from transformers import AutoModel

        self.enc = AutoModel.from_pretrained(dino_id)
        dim = int(self.enc.config.hidden_size)
        for p in self.enc.parameters():
            p.requires_grad = False
        layers = self.enc.encoder.layer
        for layer in layers[-unfreeze_last:]:
            for p in layer.parameters():
                p.requires_grad = True
        self.slot_proj = nn.Linear(dim, 256)
        self.query = nn.Parameter(torch.randn(len(TARGETS), 256) * 0.02)
        self.head = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
        self.aux = nn.Linear(256, len(TARGETS))
        self.dim = dim
        self.n_blocks = len(layers)

    def forward(self, x, m):
        b, s, c, h, w = x.shape
        inp = x.reshape(b * s, c, h, w)
        rgb = inp if c == 3 else inp.mean(1, keepdim=True).repeat(1, 3, 1, 1)
        rgb = (rgb - 0.45) / 0.22
        out = self.enc(pixel_values=rgb)
        tok = self.slot_proj(out.last_hidden_state[:, 0]).reshape(b, s, -1)
        tok = tok * m.unsqueeze(-1)
        attn = torch.einsum("bsd,td->bst", tok, self.query) / 16.0
        attn = attn.masked_fill(m.unsqueeze(-1) == 0, -1e4)
        wgt = torch.softmax(attn, dim=1)
        pooled = torch.einsum("bst,bsd->btd", wgt, tok)
        main = self.head(pooled).squeeze(-1)
        aux = self.aux(tok.mean(1))
        return main, aux


def macro_auc(y, p) -> float:
    scores = []
    for j in range(y.shape[1]):
        if len(np.unique((y[:, j] > 0.5).astype(int))) < 2:
            continue
        try:
            scores.append(roc_auc_score((y[:, j] > 0.5).astype(int), p[:, j]))
        except Exception:
            continue
    return float(np.mean(scores)) if scores else float("nan")


def _amp():
    cuda = torch.cuda.is_available()
    try:
        autocast = lambda: torch.amp.autocast("cuda", enabled=cuda)
        scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    except Exception:
        autocast = lambda: torch.cuda.amp.autocast(enabled=cuda)
        scaler = torch.cuda.amp.GradScaler(enabled=cuda)
    return autocast, scaler


def train_one_fold(cfg: TrainCfg, commit=None) -> dict:
    t0 = time.time()
    set_seed(cfg.seed + cfg.fold)
    cache_dir = Path(cfg.cache_dir)
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    man, x, mask, y, uids = load_cache(cache_dir)
    log(f"cache n={len(uids)} shape={tuple(x.shape)} recipe={man.get('recipe')}", t0)

    groups = np.array([report_group(u) for u in uids])
    gkf = GroupKFold(N_FOLDS)
    folds = list(gkf.split(np.arange(len(uids)), groups=groups))
    tr_idx, va_idx = folds[int(cfg.fold)]
    if cfg.smoke:
        tr_idx = tr_idx[:80]
        va_idx = va_idx[:40]
        log("SMOKE subset on", t0)

    ds_tr = CacheDS(x, mask, y, uids, tr_idx, augment=bool(cfg.augment) and not cfg.smoke)
    ds_va = CacheDS(x, mask, y, uids, va_idx, augment=False)
    log(f"fold{cfg.fold} train={len(ds_tr)} val={len(ds_va)}", t0)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"device={dev} cuda={torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'}", t0)
    net = PartialFT(cfg.dino_id, cfg.unfreeze_last, dropout=cfg.dropout).to(dev)
    log(f"dinov2 dim={net.dim} blocks={net.n_blocks} unfrozen_last={cfg.unfreeze_last} mixup={cfg.mixup} drop={cfg.dropout}", t0)
    params = [
        {"params": [p for n, p in net.named_parameters() if p.requires_grad and n.startswith("enc.")], "lr": cfg.lr_backbone},
        {"params": [p for n, p in net.named_parameters() if p.requires_grad and not n.startswith("enc.")], "lr": cfg.lr_head},
    ]
    opt = torch.optim.AdamW(params, weight_decay=cfg.weight_decay)
    autocast, scaler = _amp()

    last_p = out / "last.pt"
    best_p = out / "best.pt"
    hist_p = out / "history.json"
    start_epoch = 0
    best = -1.0
    bad = 0
    history = []
    if cfg.resume and last_p.is_file():
        ckpt = torch.load(last_p, map_location=dev, weights_only=False)
        net.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["opt"])
        if "scaler" in ckpt:
            try:
                scaler.load_state_dict(ckpt["scaler"])
            except Exception:
                pass
        start_epoch = int(ckpt["epoch"]) + 1
        best = float(ckpt.get("best", -1.0))
        bad = int(ckpt.get("bad", 0))
        history = list(ckpt.get("history", []))
        log(f"resume epoch={start_epoch} best={best:.4f} bad={bad}", t0)
    else:
        log("fresh start (no last.pt)", t0)

    def persist(epoch: int, is_best: bool, extra: dict) -> None:
        payload = {
            "model": net.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
            "epoch": epoch, "best": best, "bad": bad, "fold": cfg.fold,
            "history": history, "cfg": asdict(cfg),
        }
        torch.save(payload, last_p)
        if is_best:
            torch.save({"model": net.state_dict(), "auc": best, "epoch": epoch, "fold": cfg.fold}, best_p)
        (out / "receipt.json").write_text(json.dumps(extra, indent=2) + "\n", encoding="utf-8")
        hist_p.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
        if commit is not None:
            commit()

    if start_epoch >= cfg.epochs:
        receipt = {"fold": cfg.fold, "best_val_auc": best, "status": "already_done", "epochs_done": start_epoch, "seconds": time.time() - t0}
        (out / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        log(json.dumps(receipt), t0)
        return receipt

    nw = 0 if cfg.num_workers <= 0 else cfg.num_workers
    for ep in range(start_epoch, cfg.epochs):
        net.train()
        losses = []
        loader = DataLoader(ds_tr, batch_size=cfg.batch, shuffle=True, num_workers=nw, pin_memory=dev.type == "cuda", drop_last=False)
        for x_b, m_b, t_b, _uid in loader:
            x_b, m_b, t_b = x_b.to(dev, non_blocking=True), m_b.to(dev, non_blocking=True), t_b.to(dev, non_blocking=True)
            if cfg.mixup > 0:
                lam = float(np.random.beta(cfg.mixup, cfg.mixup))
                perm = torch.randperm(x_b.size(0), device=x_b.device)
                x_b = lam * x_b + (1.0 - lam) * x_b[perm]
                t_b = lam * t_b + (1.0 - lam) * t_b[perm]
            opt.zero_grad(set_to_none=True)
            with autocast():
                logit, aux = net(x_b, m_b)
                loss = F.binary_cross_entropy_with_logits(logit, t_b)
                loss = loss + AUX_W * F.binary_cross_entropy_with_logits(aux, t_b)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        net.eval()
        ys, ps, vuids = [], [], []
        with torch.no_grad():
            for x_b, m_b, t_b, uid_b in DataLoader(ds_va, batch_size=cfg.batch, shuffle=False, num_workers=nw):
                x_b, m_b = x_b.to(dev, non_blocking=True), m_b.to(dev, non_blocking=True)
                with autocast():
                    pred = torch.sigmoid(net(x_b, m_b)[0]).float().cpu().numpy()
                ys.append(t_b.numpy())
                ps.append(pred)
                vuids.extend(list(uid_b))
        yv = np.concatenate(ys)
        pv = np.concatenate(ps)
        auc = macro_auc(yv, pv)
        mean_loss = float(np.mean(losses)) if losses else float("nan")
        row = {"epoch": ep, "loss": mean_loss, "val_auc": auc, "seconds": time.time() - t0}
        history.append(row)
        log(f"ep{ep} loss={mean_loss:.4f} val_auc={auc:.4f}", t0)
        is_best = bool(math.isfinite(auc) and auc > best)
        if is_best:
            best = float(auc)
            bad = 0
            pd.DataFrame(pv, columns=TARGETS).assign(StudyInstanceUID=vuids).to_csv(out / "val_preds.csv", index=False)
        else:
            bad += 1
        receipt = {
            "fold": cfg.fold, "best_val_auc": best, "last_epoch": ep, "last_val_auc": auc,
            "patience_bad": bad, "smoke": cfg.smoke, "img": IMG, "unfreeze_last": cfg.unfreeze_last,
            "epochs_target": cfg.epochs, "n_train": len(ds_tr), "n_val": len(ds_va), "batch": cfg.batch,
            "gpu": torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu",
            "resumed_from": start_epoch, "seconds": time.time() - t0, "status": "running",
            "mixup": cfg.mixup, "dropout": cfg.dropout, "augment": cfg.augment,
        }
        persist(ep, is_best, receipt)
        if bad >= cfg.patience:
            log(f"early stop patience={cfg.patience} best={best:.4f}", t0)
            receipt["status"] = "early_stop"
            (out / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
            if commit is not None:
                commit()
            break
    else:
        receipt = json.loads((out / "receipt.json").read_text(encoding="utf-8"))
        receipt["status"] = "complete"
        (out / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        if commit is not None:
            commit()

    log(json.dumps(receipt), t0)
    return receipt
''', encoding="utf-8")
print("train_core", Path("/content/train_core.py").stat().st_size)


train_core 12775


In [ ]:
import importlib
import json
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import train_core
importlib.reload(train_core)
from train_core import TrainCfg, train_one_fold

CACHE = next(Path("/content/v28_kaggle_out").rglob("manifest.json")).parent
ROOT = Path("/content/drive/MyDrive/rsna-knee-v28/checkpoints")
jobs = []
for fold in (2, 3, 4):
    jobs.append(TrainCfg(
        cache_dir=str(CACHE),
        out_dir=str(ROOT / f"fold{fold}_u6aug"),
        fold=fold,
        epochs=16,
        batch=16,
        lr_head=1.5e-4,
        lr_backbone=1.5e-5,
        weight_decay=2e-4,
        unfreeze_last=6,
        patience=6,
        resume=True,
        num_workers=2,
        dino_id="facebook/dinov2-small",
        augment=True,
        mixup=0.2,
        dropout=0.35,
        tta=True,
    ))
for fold in (2, 3, 4):
    jobs.append(TrainCfg(
        cache_dir=str(CACHE),
        out_dir=str(ROOT / f"fold{fold}_base"),
        fold=fold,
        epochs=12,
        batch=8,
        lr_head=1.5e-4,
        lr_backbone=1e-5,
        weight_decay=2e-4,
        unfreeze_last=4,
        patience=5,
        resume=True,
        num_workers=2,
        dino_id="facebook/dinov2-base",
        augment=True,
        mixup=0.0,
        dropout=0.3,
        tta=True,
    ))

summary = []
for cfg in jobs:
    print("START", cfg.out_dir, flush=True)
    rec = train_one_fold(cfg)
    rec["out_dir"] = cfg.out_dir
    summary.append(rec)
    (ROOT / "train_234_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print("DONE", rec.get("fold"), rec.get("best_val_auc"), rec.get("status"), flush=True)
print(json.dumps(summary, indent=2))


START /content/drive/MyDrive/rsna-knee-v28/checkpoints/fold2_u6aug
[    0.0s] cache n=4407 shape=(4407, 3, 3, 224, 224) recipe=v28-224-3slice-3fs-planes
[    0.0s] fold2 train=3526 val=881
[    0.0s] device=cuda cuda=Tesla T4
[    0.2s] dinov2 dim=384 blocks=12 unfrozen_last=6 mixup=0.2 drop=0.35
[    0.2s] fresh start (no last.pt)
[   39.5s] ep0 loss=0.7770 val_auc=0.7112
[   80.3s] ep1 loss=0.7368 val_auc=0.7345
[  120.2s] ep2 loss=0.7225 val_auc=0.7472
[  160.7s] ep3 loss=0.7146 val_auc=0.7508
[  200.7s] ep4 loss=0.7096 val_auc=0.7583
[  241.1s] ep5 loss=0.7015 val_auc=0.7731
[  281.3s] ep6 loss=0.6896 val_auc=0.7704
[  321.4s] ep7 loss=0.6780 val_auc=0.7857
[  361.1s] ep8 loss=0.6694 val_auc=0.7869
[  401.4s] ep9 loss=0.6618 val_auc=0.8030
[  442.3s] ep10 loss=0.6475 val_auc=0.8017
[  482.1s] ep11 loss=0.6371 val_auc=0.8059
[  522.1s] ep12 loss=0.6284 val_auc=0.8099
[  562.2s] ep13 loss=0.6207 val_auc=0.8045
[  602.5s] ep14 loss=0.6173 val_auc=0.8074
[  642.7s] ep15 loss=0.6042 val

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader
import train_core as tc

CACHE = next(Path("/content/v28_kaggle_out").rglob("manifest.json")).parent
man, x, mask, y, uids = tc.load_cache(CACHE)
groups = np.array([tc.report_group(u) for u in uids])
_, va = list(GroupKFold(5).split(np.arange(len(uids)), groups=groups))[0]
ds = tc.CacheDS(x, mask, y, uids, va, augment=False)
dev = torch.device("cuda")
net = tc.PartialFT("facebook/dinov2-small", 6, 0.35).to(dev)
ckpt = torch.load("/content/drive/MyDrive/rsna-knee-v28/checkpoints/fold0_u6aug/best.pt", map_location=dev, weights_only=False)
net.load_state_dict(ckpt["model"])
net.eval()
ys, ps, ps_tta = [], [], []
with torch.no_grad():
    for xb, mb, tb, _ in DataLoader(ds, batch_size=16):
        xb, mb = xb.to(dev), mb.to(dev)
        p = torch.sigmoid(net(xb, mb)[0])
        pf = torch.sigmoid(net(torch.flip(xb, dims=[-1]), mb)[0])
        ys.append(tb.numpy())
        ps.append(p.float().cpu().numpy())
        ps_tta.append((0.5 * (p + pf)).float().cpu().numpy())
yv = np.concatenate(ys)
print("plain", tc.macro_auc(yv, np.concatenate(ps)), "tta", tc.macro_auc(yv, np.concatenate(ps_tta)), "ckpt", ckpt.get("auc"), ckpt.get("epoch"))


plain 0.8104606101283882 tta 0.8157210670167906 ckpt 0.8104403324484873 11


In [ ]:
from pathlib import Path
p = Path("/content/train_core.py")
t = p.read_text(encoding="utf-8")
if "tta: bool" not in t:
    t = t.replace("dropout: float = 0.35\n", "dropout: float = 0.35\n    tta: bool = True\n")
old = """                with autocast():
                    pred = torch.sigmoid(net(x_b, m_b)[0]).float().cpu().numpy()"""
new = """                with autocast():
                    pred = torch.sigmoid(net(x_b, m_b)[0])
                    if getattr(cfg, "tta", False):
                        pred = 0.5 * (pred + torch.sigmoid(net(torch.flip(x_b, dims=[-1]), m_b)[0]))
                    pred = pred.float().cpu().numpy()"""
if old not in t:
    raise SystemExit("val pred block not found")
p.write_text(t.replace(old, new), encoding="utf-8")
print("patched tta")


patched tta


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import train_core as tc

def aucs(df, y, uids, va_uids):
    m = df.set_index("StudyInstanceUID").reindex(va_uids)
    out = {}
    for t in tc.TARGETS:
        yt = (y[:, tc.TARGETS.index(t)] > 0.5).astype(int)
        if len(np.unique(yt)) < 2:
            out[t] = float("nan")
            continue
        out[t] = float(roc_auc_score(yt, m[t].to_numpy()))
    out["macro"] = float(np.nanmean(list(out.values())))
    return out

CACHE = next(Path("/content/v28_kaggle_out").rglob("manifest.json")).parent
man, x, mask, y, uids = tc.load_cache(CACHE)
from sklearn.model_selection import GroupKFold
groups = np.array([tc.report_group(u) for u in uids])
_, va = list(GroupKFold(5).split(np.arange(len(uids)), groups=groups))[0]
va_uids = [uids[i] for i in va]
yv = y[va]

root = Path("/content/drive/MyDrive/rsna-knee-v28/checkpoints")
for name in ("fold0", "fold0_u6aug", "fold0_base"):
    p = root / name / "val_preds.csv"
    if not p.is_file():
        print(name, "missing")
        continue
    df = pd.read_csv(p)
    a = aucs(df, yv, uids, va_uids)
    print(name, {k: round(v, 4) for k, v in a.items()})

dfs = []
for name in ("fold0_u6aug", "fold0_base"):
    p = root / name / "val_preds.csv"
    if p.is_file():
        dfs.append(pd.read_csv(p).set_index("StudyInstanceUID")[tc.TARGETS])
if len(dfs) == 2:
    blend = 0.5 * (dfs[0].rank(pct=True) + dfs[1].rank(pct=True))
    blend = blend.reset_index()
    a = aucs(blend, yv, uids, va_uids)
    print("rankblend", {k: round(v, 4) for k, v in a.items()})


fold0 {'ACL': 0.7785, 'MCL': 0.7676, 'Medial Meniscus': 0.7847, 'Lateral Meniscus': 0.7294, 'Medial OA': 0.827, 'Lateral OA': 0.7664, 'PF OA': 0.7632, 'Effusion': 0.8352, 'Synovitis': 0.8567, "Baker's": 0.8764, 'Contusion': 0.8013, 'Fracture': 0.7802, 'macro': 0.7972}
fold0_u6aug {'ACL': 0.8199, 'MCL': 0.7853, 'Medial Meniscus': 0.8018, 'Lateral Meniscus': 0.7426, 'Medial OA': 0.8363, 'Lateral OA': 0.7795, 'PF OA': 0.7883, 'Effusion': 0.8328, 'Synovitis': 0.8633, "Baker's": 0.8647, 'Contusion': 0.8178, 'Fracture': 0.793, 'macro': 0.8104}
fold0_base {'ACL': 0.8335, 'MCL': 0.7594, 'Medial Meniscus': 0.7962, 'Lateral Meniscus': 0.7408, 'Medial OA': 0.8325, 'Lateral OA': 0.7745, 'PF OA': 0.7924, 'Effusion': 0.8316, 'Synovitis': 0.8769, "Baker's": 0.8712, 'Contusion': 0.8219, 'Fracture': 0.836, 'macro': 0.8139}
rankblend {'ACL': 0.8522, 'MCL': 0.7919, 'Medial Meniscus': 0.8187, 'Lateral Meniscus': 0.7546, 'Medial OA': 0.8478, 'Lateral OA': 0.79, 'PF OA': 0.8088, 'Effusion': 0.8497, 'Synovit

In [ ]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
import train_core as tc
importlib.reload(tc)

def tta_preds(ckpt_path, dino_id, unfreeze, dropout, ds, dev):
    net = tc.PartialFT(dino_id, unfreeze, dropout).to(dev)
    ckpt = torch.load(ckpt_path, map_location=dev, weights_only=False)
    net.load_state_dict(ckpt["model"])
    net.eval()
    ys, ps, uout = [], [], []
    with torch.no_grad():
        for xb, mb, tb, uid in DataLoader(ds, batch_size=8 if "base" in dino_id else 16):
            xb, mb = xb.to(dev), mb.to(dev)
            p = torch.sigmoid(net(xb, mb)[0])
            pf = torch.sigmoid(net(torch.flip(xb, dims=[-1]), mb)[0])
            ys.append(tb.numpy())
            ps.append((0.5 * (p + pf)).float().cpu().numpy())
            uout.extend(list(uid))
    return np.concatenate(ys), np.concatenate(ps), uout

CACHE = next(Path("/content/v28_kaggle_out").rglob("manifest.json")).parent
man, x, mask, y, uids = tc.load_cache(CACHE)
groups = np.array([tc.report_group(u) for u in uids])
folds = list(GroupKFold(5).split(np.arange(len(uids)), groups=groups))
_, va = folds[1]
ds = tc.CacheDS(x, mask, y, uids, va, augment=False)
dev = torch.device("cuda")
root = Path("/content/drive/MyDrive/rsna-knee-v28/checkpoints")
y1, p1, u1 = tta_preds(root / "fold1_u6aug/best.pt", "facebook/dinov2-small", 6, 0.35, ds, dev)
y2, p2, u2 = tta_preds(root / "fold1_base/best.pt", "facebook/dinov2-base", 4, 0.3, ds, dev)
assert u1 == u2
d1 = pd.DataFrame(p1, columns=tc.TARGETS)
d2 = pd.DataFrame(p2, columns=tc.TARGETS)
blend = 0.5 * (d1.rank(pct=True) + d2.rank(pct=True))
print("fold1 tta_small", tc.macro_auc(y1, p1))
print("fold1 tta_base", tc.macro_auc(y2, p2))
print("fold1 tta_rankblend", tc.macro_auc(y1, blend.to_numpy()))
print("gate", tc.macro_auc(y1, blend.to_numpy()) >= 0.83)


fold1 tta_small 0.800193618050141
fold1 tta_base 0.7960207439541761
fold1 tta_rankblend 0.813498031614742
gate False


In [ ]:
import importlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
import train_core as tc
importlib.reload(tc)

def tta_fold(ckpt, dino_id, unfreeze, dropout, ds, dev, bs):
    net = tc.PartialFT(dino_id, unfreeze, dropout).to(dev)
    payload = torch.load(ckpt, map_location=dev, weights_only=False)
    net.load_state_dict(payload["model"])
    net.eval()
    ys, ps, uids = [], [], []
    with torch.no_grad():
        for xb, mb, tb, uid in DataLoader(ds, batch_size=bs):
            xb, mb = xb.to(dev), mb.to(dev)
            p = torch.sigmoid(net(xb, mb)[0])
            pf = torch.sigmoid(net(torch.flip(xb, dims=[-1]), mb)[0])
            ys.append(tb.numpy())
            ps.append((0.5 * (p + pf)).float().cpu().numpy())
            uids.extend(list(uid))
    return np.concatenate(ys), np.concatenate(ps), uids

CACHE = next(Path("/content/v28_kaggle_out").rglob("manifest.json")).parent
ROOT = Path("/content/drive/MyDrive/rsna-knee-v28/checkpoints")
OUT = Path("/content/drive/MyDrive/rsna-knee-v28/oof")
OUT.mkdir(parents=True, exist_ok=True)
man, x, mask, y, uids = tc.load_cache(CACHE)
groups = np.array([tc.report_group(u) for u in uids])
folds = list(GroupKFold(5).split(np.arange(len(uids)), groups=groups))
dev = torch.device("cuda")

specs = {
    "small": dict(tag="u6aug", dino="facebook/dinov2-small", unfreeze=6, drop=0.35, bs=16),
    "base": dict(tag="base", dino="facebook/dinov2-base", unfreeze=4, drop=0.3, bs=8),
}
oof = {}
for fam, sp in specs.items():
    rows = []
    for k, (_, va) in enumerate(folds):
        ck = ROOT / f"fold{k}_{sp['tag']}" / "best.pt"
        ds = tc.CacheDS(x, mask, y, uids, va, augment=False)
        yv, pv, uv = tta_fold(ck, sp["dino"], sp["unfreeze"], sp["drop"], ds, dev, sp["bs"])
        df = pd.DataFrame(pv, columns=tc.TARGETS)
        df.insert(0, "StudyInstanceUID", uv)
        df["fold"] = k
        rows.append(df)
        print(fam, "fold", k, "auc", round(tc.macro_auc(yv, pv), 4), flush=True)
    oof[fam] = pd.concat(rows, ignore_index=True)
    oof[fam].to_csv(OUT / f"oof_{fam}.csv", index=False)

idx = oof["small"].set_index("StudyInstanceUID")
bdx = oof["base"].set_index("StudyInstanceUID").reindex(idx.index)
blend = 0.5 * (idx[tc.TARGETS].rank(pct=True) + bdx[tc.TARGETS].rank(pct=True))
blend = blend.reset_index()
blend.to_csv(OUT / "oof_blend.csv", index=False)

lab = pd.DataFrame(y, columns=tc.TARGETS)
lab["StudyInstanceUID"] = uids
lab = lab.set_index("StudyInstanceUID")

def score(df, name):
    m = df.set_index("StudyInstanceUID")
    per = {}
    for t in tc.TARGETS:
        yt = (lab.loc[m.index, t].to_numpy() > 0.5).astype(int)
        per[t] = float(roc_auc_score(yt, m[t].to_numpy())) if len(np.unique(yt)) > 1 else float("nan")
    per["macro"] = float(np.nanmean(list(per.values())))
    print(name, {k: round(v, 4) for k, v in per.items()}, flush=True)
    return per

rec = {
    "small": score(oof["small"], "OOF_small"),
    "base": score(oof["base"], "OOF_base"),
    "blend": score(blend, "OOF_blend"),
}
(OUT / "oof_receipt.json").write_text(json.dumps(rec, indent=2) + "\n")
print("wrote", OUT)


small fold 0 auc 0.8157
small fold 1 auc 0.8002
small fold 2 auc 0.8099
small fold 3 auc 0.8139
small fold 4 auc 0.8109
base fold 0 auc 0.8139
base fold 1 auc 0.796
base fold 2 auc 0.8094
base fold 3 auc 0.8055
base fold 4 auc 0.7999
OOF_small {'ACL': 0.8289, 'MCL': 0.7478, 'Medial Meniscus': 0.7946, 'Lateral Meniscus': 0.7394, 'Medial OA': 0.8445, 'Lateral OA': 0.7927, 'PF OA': 0.7821, 'Effusion': 0.8181, 'Synovitis': 0.8475, "Baker's": 0.8509, 'Contusion': 0.7938, 'Fracture': 0.8116, 'macro': 0.8043}
OOF_base {'ACL': 0.8144, 'MCL': 0.7284, 'Medial Meniscus': 0.792, 'Lateral Meniscus': 0.7341, 'Medial OA': 0.8359, 'Lateral OA': 0.7854, 'PF OA': 0.7711, 'Effusion': 0.8159, 'Synovitis': 0.8302, "Baker's": 0.8411, 'Contusion': 0.7959, 'Fracture': 0.8268, 'macro': 0.7976}
OOF_blend {'ACL': 0.8426, 'MCL': 0.7558, 'Medial Meniscus': 0.811, 'Lateral Meniscus': 0.7512, 'Medial OA': 0.8521, 'Lateral OA': 0.8016, 'PF OA': 0.7936, 'Effusion': 0.8329, 'Synovitis': 0.8604, "Baker's": 0.862, 'Contu